## **importing and exploration of the dataset**

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("dirty_cafe_sales.csv")

In [3]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [4]:
df.tail()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [6]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [7]:
df.duplicated().sum()

0

#### Display all the items becauese they are categorical data and we need explore them to handle them in the next steps

In [8]:
df["Item"].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

## **Handling Invalid Values**

### before the correction of data types we must handle invalid values 

In [9]:
df.replace(["ERROR", "UNKNOWN", ""], np.nan, inplace=True)

In [10]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## **Data Type Correction**

In [11]:

df[["Quantity", "Price Per Unit", "Total Spent"]] = (
    df[["Quantity", "Price Per Unit", "Total Spent"]]
    .apply(pd.to_numeric, errors="coerce")
)

In [12]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

## **Handling missing values**

### I used this formula to handle missing values  **Total Spent = Quantity * Price Per Unit.**

In [13]:
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])


### Handling Items names  

### I handled them by imputation for the most frequent items based on the price 

In [14]:
item_mode_mapping = df.dropna(subset=['Item']).groupby('Price Per Unit')['Item'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)

In [15]:
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(item_mode_mapping))

In [16]:
df.dropna(subset=['Item'], inplace=True)

### Handling Payment Method 
### I used Mode because they are categorical data

In [17]:
df["Payment Method"].unique()

array(['Credit Card', 'Cash', nan, 'Digital Wallet'], dtype=object)

In [18]:
payment_mode = df['Payment Method'].mode()[0]

In [19]:
df['Payment Method'] = df['Payment Method'].fillna(payment_mode)

### handling Location 
### I used Mode because they are categorical data

In [20]:
location_mode = df['Location'].mode()[0]

df['Location'] = df['Location'].fillna(location_mode)

### Handling Transaction Date
### I decided to delete the null values becauese we can not use mean or mode in dates

In [24]:
df.dropna(subset=['Transaction Date'], inplace=True)

## **Create New feature called "Season"**

In [26]:
def get_season(date):
    if pd.isna(date):
        return np.nan
    if date.month in [12, 1, 2]:
        return "Winter"
    elif date.month in [3, 4, 5]:
        return "Spring"
    elif date.month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["season"] = df["Transaction Date"].apply(get_season)

##	**Output Cleaned Data**

In [28]:
df.to_csv("cleaned_cafe_sales.csv", index=False)